# Cross-model replication — Codestral-22B

Same three measurements as the Qwen2.5-Coder-7B study, on the code-specialised sibling of Mistral-Small-24B:
**brain** (linear probe on frozen hidden states, CVE-grouped CV, temporal split),
**control** (same probe transferred to 226 length-matched non-security patches; length-residualised probe),
**mouth** (the same checkpoint asked directly: yes/no, expert yes/no, A/B forced choice; answer-token probabilities).
Codestral-22B-v0.1 ships a single checkpoint that is both the base and the instruct model (it has a chat template), so
`MODEL_BASE == MODEL_INSTRUCT`: one ~44 GB download, no cache deletion between the brain and mouth cells.
Self-contained (data embedded). Runtime: Colab Enterprise L4 (24 GB), ~1.5–2 h. Run cells one at a time.

In [ ]:
#@title 1. Install
!pip -q install -U transformers accelerate bitsandbytes scikit-learn scipy
import torch, numpy as np, pandas as pd, os, re, glob, json, zipfile
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch to a GPU runtime!")
if torch.cuda.is_available(): print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 136.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 149.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 150.7 MB/s eta 0:00:00
GPU: NVIDIA L4
VRAM GB: 23.7


In [ ]:
#@title 1b. Colab Enterprise: put the Hugging Face cache on the 100 GB data disk (94 GB boot disk is too small)
import os, shutil, subprocess
hub=os.path.expanduser('~/.cache/huggingface/hub')
if not os.path.islink(hub):
    shutil.rmtree(hub, ignore_errors=True); os.makedirs(os.path.dirname(hub), exist_ok=True)
    os.makedirs('/content/hf_hub', exist_ok=True); os.symlink('/content/hf_hub', hub)
print(subprocess.run(['df','-h','/','/content'],capture_output=True,text=True).stdout); print(os.path.realpath(hub))

Filesystem      Size  Used Avail Use% Mounted on
overlay          95G   66G   29G  70% /
/dev/nvme0n2     98G   12K   98G   1% /content

/content/hf_hub


In [ ]:
#@title 2. Config (Codestral-22B-v0.1 is not gated on the Hub any more; no token needed)
MODEL_BASE     = "mistralai/Codestral-22B-v0.1"
MODEL_INSTRUCT = "mistralai/Codestral-22B-v0.1"
MODEL_TAG      = "Codestral-22B"
ATTN_EAGER     = False
MAX_TOKENS   = 3000
PAIR_TOKENS  = 1400
CACHE_CVE, CACHE_CTL = "hs_cve.npz", "hs_ctl.npz"
SEED = 0
np.random.seed(SEED); torch.manual_seed(SEED)
HF_TOKEN = None   # if the repo is ever re-gated: from getpass import getpass; HF_TOKEN = getpass("HF token: ")

In [ ]:
#@title 2b. Materialize embedded data (CVE pairs + metadata)
import base64, os
_DZ='''...
APgB+AFWzAAAv+EHAAAA
'''
_MD='''...
MS0wMy0wNSxubyxPSywyNTcNCg==
'''
open('dataset.zip','wb').write(base64.b64decode(_DZ))
open('halurust_metadata.csv','wb').write(base64.b64decode(_MD))
del _DZ,_MD
print('materialized', os.path.getsize('dataset.zip'), os.path.getsize('halurust_metadata.csv'))

materialized 568875 25099


In [ ]:
#@title 2c. Materialize embedded control pairs
import base64, os
_CZ='''...
'''
open('control.zip','wb').write(base64.b64decode(_CZ))
del _CZ
print('materialized', os.path.getsize('control.zip'))

materialized 481249


In [ ]:
#@title 3. Load pairs + metadata, apply audit exclusions
if os.path.exists("dataset.zip"):
    with zipfile.ZipFile("dataset.zip") as z: z.extractall("data")
POS = sorted(glob.glob("data/**/Positive/*.rs", recursive=True))
NEG = sorted(glob.glob("data/**/Negative/*.rs", recursive=True))
assert POS and NEG, "no data found"
meta = pd.read_csv("halurust_metadata.csv")
meta["date"] = meta[["advisory_date","commit_date"]].fillna("").max(axis=1)
flagged = set(meta.loc[~meta.audit_verdict.eq("OK"), "cve"])
never_patched = set(meta.loc[meta.never_patched_upstream.eq("yes"), "cve"])
cve2 = meta.set_index("cve").to_dict("index")
def parse(fp):
    m = re.match(r"(CVE-\d{4}-\d+)_?(CWE-[\w-]+)?", os.path.basename(fp))
    return (m.group(1), m.group(2) or "") if m else (None, "")
neg_by_cve = {parse(fp)[0]: fp for fp in NEG}
samples = []
for fp in POS:
    cve, cwe = parse(fp)
    if cve is None or cve not in neg_by_cve or cve in flagged: continue
    for path, label in [(fp, 1), (neg_by_cve[cve], 0)]:
        samples.append(dict(cve=cve, cwe=cwe, label=label,
                            code=open(path, encoding="utf-8", errors="replace").read(),
                            date=str(cve2.get(cve, {}).get("date","")),
                            never_patched=cve in never_patched))
df = pd.DataFrame(samples)
pair_cves = sorted({c for c,g in df.groupby("cve") if set(g.label)=={0,1}})
eqlen = set()
for cve,g in df.groupby("cve"):
    if set(g.label)=={0,1}:
        lv=len(g.loc[g.label==1,"code"].iloc[0]); lf=len(g.loc[g.label==0,"code"].iloc[0])
        if abs(lv-lf)<=20: eqlen.add(cve)
print(f"pairs: {len(pair_cves)}  samples: {len(df)}  eq-len pairs: {len(eqlen)}  post-2024: {df[df.date>='2024'].cve.nunique()}")

def pair_outcomes(scores, subset=None):
    """per-pair: 1.0 probe ranks vuln higher, 0.5 tie, 0.0 wrong. returns dict cve->outcome"""
    out={}
    for cve,g in df.assign(s=scores).groupby("cve"):
        if subset is not None and cve not in subset: continue
        if set(g.label)=={0,1}:
            sv=g.loc[g.label==1,"s"].iloc[0]; sf=g.loc[g.label==0,"s"].iloc[0]
            out[cve]= 1.0 if sv>sf else (0.5 if sv==sf else 0.0)
    return out
def pairwise_acc(scores, subset=None):
    o=pair_outcomes(scores,subset); return sum(o.values())/len(o) if o else float("nan")

pairs: 228  samples: 460  eq-len pairs: 36  post-2024: 56


In [ ]:
#@title 3b. Load control (non-security) pairs
with zipfile.ZipFile("control.zip") as z: z.extractall("ctl")
cmeta = pd.read_csv("ctl/control_meta.csv")
rows=[]
for _,r in cmeta.iterrows():
    for sub,label in [("Before",1),("After",0)]:   # label 1 = "before" (plays the role of 'vulnerable' twin)
        rows.append(dict(cve=r["id"], cwe="", label=label, code=open(f"ctl/{sub}/{r['id']}.rs",encoding="utf-8",errors="replace").read(),
                         date=str(r["date"]), never_patched=False, repo=r["repo"], cat=r["cat"]))
dfc = pd.DataFrame(rows)
ctl_eqlen=set()
for pid,g in dfc.groupby("cve"):
    lb=len(g.loc[g.label==1,"code"].iloc[0]); la=len(g.loc[g.label==0,"code"].iloc[0])
    if abs(lb-la)<=20: ctl_eqlen.add(pid)
print(f"control pairs: {dfc.cve.nunique()}  samples: {len(dfc)}  repos: {dfc.repo.nunique()}  non-fix commits: {(cmeta['cat']=='nonfix').sum()}  eq-len: {len(ctl_eqlen)}")
print("CVE   frac fixed-longer :", np.mean([len(g.loc[g.label==0,'code'].iloc[0])>len(g.loc[g.label==1,'code'].iloc[0]) for c,g in df.groupby('cve') if set(g.label)=={0,1}]).round(3))
print("CTL   frac after-longer :", np.mean([len(g.loc[g.label==0,'code'].iloc[0])>len(g.loc[g.label==1,'code'].iloc[0]) for c,g in dfc.groupby('cve')]).round(3))

def pair_outcomes_df(d, scores, subset=None):
    out={}
    for k,g in d.assign(s=scores).groupby("cve"):
        if subset is not None and k not in subset: continue
        if set(g.label)=={0,1}:
            sv=g.loc[g.label==1,"s"].iloc[0]; sf=g.loc[g.label==0,"s"].iloc[0]
            out[k]= 1.0 if sv>sf else (0.5 if sv==sf else 0.0)
    return out
def pw(d, scores, subset=None):
    o=pair_outcomes_df(d,scores,subset); return sum(o.values())/len(o) if o else float("nan")
rng = np.random.default_rng(0)
def boot_ci(outcomes, iters=5000):
    vals=np.array(list(outcomes.values())); n=len(vals)
    bs=np.array([vals[rng.integers(0,n,n)].mean() for _ in range(iters)])
    return vals.mean(), np.percentile(bs,2.5), np.percentile(bs,97.5), n
# length floor on control: 'shorter twin is before'
ctl_len_scores = -dfc.code.str.len().values.astype(float)
print(f"control length rule ('shorter is before'): non-fix {pw(dfc, ctl_len_scores, set(dfc[dfc['cat']=='nonfix'].cve)):.3f}   bug-fix {pw(dfc, ctl_len_scores, set(dfc[dfc['cat']=='bugfix'].cve)):.3f}")

control pairs: 268  samples: 536  repos: 128  non-fix commits: 226  eq-len: 39
CVE   frac fixed-longer : 0.794
CTL   frac after-longer : 0.765
control length rule ('shorter is before'): non-fix 0.801   bug-fix 0.631


In [ ]:
#@title 4. Floors (grouped-CV direction)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
def grouped_cv_scores(X):
    X=np.asarray(X,dtype=float).reshape(len(df),-1); s=np.zeros(len(df))
    for tr,te in GroupKFold(5).split(X, df.label, groups=df.cve):
        m=LogisticRegression(max_iter=3000).fit(X[tr], df.label.iloc[tr]); s[te]=m.predict_proba(X[te])[:,1]
    return s
len_scores = grouped_cv_scores(df.code.str.len().values[:,None])
len_out = pair_outcomes(len_scores)
print(f"length floor: AUC {roc_auc_score(df.label,len_scores):.3f}  pairwise {sum(len_out.values())/len(len_out):.3f}  eq-len {pairwise_acc(len_scores,eqlen):.3f}")

length floor: AUC 0.544  pairwise 0.805  eq-len 0.681


In [10]:
#@title 5. Extract mean-pooled hidden states from the BASE model: CVE set + control set
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
kw = dict(device_map="auto", torch_dtype=torch.float16,
          quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), token=HF_TOKEN)
if ATTN_EAGER: kw["attn_implementation"] = "eager"
tok = AutoTokenizer.from_pretrained(MODEL_BASE, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_BASE, **kw); model.eval()
print(MODEL_BASE, "layers:", model.config.num_hidden_layers, "hidden:", model.config.hidden_size)
def extract(codes, cache):
    if os.path.exists(cache): return np.load(cache)["mean"]
    out=[]
    with torch.no_grad():
        for i,c in enumerate(codes):
            enc=tok(c, return_tensors="pt", truncation=True, max_length=MAX_TOKENS).to(model.device)
            hs=model(**enc, output_hidden_states=True).hidden_states
            out.append(torch.stack([h[0].mean(0) for h in hs]).float().cpu().numpy())
            if i%100==0: print(f"{cache} {i}/{len(codes)}")
    H=np.stack(out); np.savez_compressed(cache, mean=H); return H
H_cve = extract(list(df.code), CACHE_CVE)
H_ctl = extract(list(dfc.code), CACHE_CTL)
N,L1,d = H_cve.shape; print("CVE",H_cve.shape,"CTL",H_ctl.shape)
del model; torch.cuda.empty_cache()
# base == instruct for Codestral: keep the disk cache so the mouth cell reuses the same download

[NOTE: this cell's real output — execution [10], 33 min on the L4 — was overwritten by an accidental 0.6 s re-run
(execution [11]) that raised `ValueError: Some modules are dispatched on the CPU or the disk` because the GPU still
held the just-freed model. The lines below are reconstructed from the live view of execution [10]; the cached
hs_cve.npz / hs_ctl.npz from that run fed cells 6–7 unchanged.]
mistralai/Codestral-22B-v0.1 layers: 56 hidden: 6144
hs_cve.npz 0/460
hs_cve.npz 100/460
hs_cve.npz 200/460
hs_cve.npz 300/460
hs_cve.npz 400/460
hs_ctl.npz 0/536
hs_ctl.npz 100/536
hs_ctl.npz 200/536
hs_ctl.npz 300/536
hs_ctl.npz 400/536
hs_ctl.npz 500/536
CVE (460, 57, 6144) CTL (536, 57, 6144)


**Run note (Sep 22 2026).** Cells were run one at a time on Colab Enterprise (us-east4, g2-standard-4 / L4). Cell 5's original output (33 min) was lost to an accidental re-run that failed instantly; the reconstructed lines above are what it printed. `gc` check before the mouth: `alloc GB 0.01 reserved GB 0.2`. Total wall time ≈ 1 h 50 min.

In [12]:
#@title 6. CVE probes per layer (grouped 5-fold CV) -> best layer
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
def mk(): return make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
def cv_scores(H, d_, l):
    X=H[:,l,:]; s=np.zeros(len(d_))
    for tr,te in GroupKFold(5).split(X, d_.label, groups=d_.cve):
        s[te]=mk().fit(X[tr], d_.label.iloc[tr]).predict_proba(X[te])[:,1]
    return s
layer_scores={l:cv_scores(H_cve, df, l) for l in range(L1)}
for l in range(L1): print(f"L{l:2d}: pairwise {pw(df,layer_scores[l]):.3f}  eq-len {pw(df,layer_scores[l],eqlen):.3f}  AUC {roc_auc_score(df.label,layer_scores[l]):.3f}")
best_layer = max(layer_scores, key=lambda l: pw(df,layer_scores[l]))
probe_scores = layer_scores[best_layer]
cve_out = pair_outcomes_df(df, probe_scores)
m,lo,hi,n = boot_ci(cve_out)
print(f"BEST L{best_layer}: CVE pairwise {m:.3f} CI[{lo:.3f},{hi:.3f}] n={n}  AUC {roc_auc_score(df.label,probe_scores):.3f}")

L 0: pairwise 0.592  eq-len 0.569  AUC 0.534
L 1: pairwise 0.621  eq-len 0.653  AUC 0.546
L 2: pairwise 0.656  eq-len 0.653  AUC 0.560
L 3: pairwise 0.643  eq-len 0.625  AUC 0.565
L 4: pairwise 0.700  eq-len 0.681  AUC 0.566
L 5: pairwise 0.671  eq-len 0.597  AUC 0.567
L 6: pairwise 0.686  eq-len 0.639  AUC 0.570
L 7: pairwise 0.724  eq-len 0.667  AUC 0.579
L 8: pairwise 0.704  eq-len 0.639  AUC 0.581
L 9: pairwise 0.693  eq-len 0.653  AUC 0.571
L10: pairwise 0.662  eq-len 0.569  AUC 0.565
L11: pairwise 0.691  eq-len 0.639  AUC 0.565
L12: pairwise 0.704  eq-len 0.569  AUC 0.566
L13: pairwise 0.702  eq-len 0.611  AUC 0.565
L14: pairwise 0.730  eq-len 0.514  AUC 0.569
L15: pairwise 0.702  eq-len 0.583  AUC 0.570
L16: pairwise 0.691  eq-len 0.556  AUC 0.580
L17: pairwise 0.728  eq-len 0.597  AUC 0.583
L18: pairwise 0.726  eq-len 0.542  AUC 0.593
L19: pairwise 0.748  eq-len 0.694  AUC 0.602
L20: pairwise 0.754  eq-len 0.708  AUC 0.598
L21: pairwise 0.774  eq-len 0.708  AUC 0.607
L22: pairw

In [13]:
#@title 7. Temporal split + control transfer + length-residualised probe
from sklearn.linear_model import LinearRegression
results = dict(model=MODEL_TAG, base=MODEL_BASE, instruct=MODEL_INSTRUCT, layers=int(L1-1), best_layer=int(best_layer))
results["probe"] = dict(pairwise=[m,lo,hi,n], auc=float(roc_auc_score(df.label, probe_scores)), eqlen=pw(df,probe_scores,eqlen))
# temporal
pre  = df.index[df.date <  "2024"].to_numpy(); post = df.index[df.date >= "2024"].to_numpy()
X = H_cve[:, best_layer, :]
pipe = mk().fit(X[pre], df.label.iloc[pre]); s_post = np.zeros(len(df)); s_post[post] = pipe.predict_proba(X[post])[:,1]
post_cves = set(df.iloc[post].cve); t_out = {c:v for c,v in pair_outcomes_df(df, s_post).items() if c in post_cves}
mt,lot,hit,nt = boot_ci(t_out); results["temporal"]=[mt,lot,hit,nt]
print(f"TEMPORAL (train <2024, test 2024+): {mt:.3f} CI[{lot:.3f},{hit:.3f}] n={nt}")
# control transfer
probe_full = mk().fit(H_cve[:,best_layer,:], df.label)
ctl_scores = probe_full.predict_proba(H_ctl[:,best_layer,:])[:,1]
ctl_out = pair_outcomes_df(dfc, ctl_scores)
cat_of = dfc.drop_duplicates("cve").set_index("cve")["cat"].to_dict()
ctl_nonfix = {k:v for k,v in ctl_out.items() if cat_of[k]=="nonfix"}; ctl_bugfix = {k:v for k,v in ctl_out.items() if cat_of[k]=="bugfix"}
mc,loc,hic,nc = boot_ci(ctl_nonfix); mbf,lobf,hibf,nbf = boot_ci(ctl_bugfix)
nonfix_ids=set(ctl_nonfix)
print(f"CONTROL non-security patches: probe picks 'before' {mc:.3f} CI[{loc:.3f},{hic:.3f}] n={nc}  (length rule {pw(dfc,ctl_len_scores,nonfix_ids):.3f})")
print(f"CONTROL bug-fix commits: {mbf:.3f} CI[{lobf:.3f},{hibf:.3f}] n={nbf}")
results["control"]=dict(nonfix=[mc,loc,hic,nc], bugfix=[mbf,lobf,hibf,nbf], len_rule=pw(dfc,ctl_len_scores,nonfix_ids), gap=m-mc)
# residualised
def resid_cv(H, d_, l):
    Xl=H[:,l,:]; z=np.log1p(d_.code.str.len().values)[:,None]; s=np.zeros(len(d_))
    for tr,te in GroupKFold(5).split(Xl, d_.label, groups=d_.cve):
        reg=LinearRegression().fit(z[tr], Xl[tr]); s[te]=mk().fit(Xl[tr]-reg.predict(z[tr]), d_.label.iloc[tr]).predict_proba(Xl[te]-reg.predict(z[te]))[:,1]
    return s
res_scores = resid_cv(H_cve, df, best_layer); mr,lor,hir,nr = boot_ci(pair_outcomes_df(df,res_scores))
print(f"RESIDUALISED probe: {mr:.3f} CI[{lor:.3f},{hir:.3f}]  AUC {roc_auc_score(df.label,res_scores):.3f}")
results["residualised"]=[mr,lor,hir,nr]
results["length_floor"]=dict(pairwise=sum(len_out.values())/len(len_out), eqlen=pairwise_acc(len_scores,eqlen))

TEMPORAL (train <2024, test 2024+): 0.830 CI[0.732,0.920] n=56
CONTROL non-security patches: probe picks 'before' 0.637 CI[0.575,0.697] n=226  (length rule 0.801)
CONTROL bug-fix commits: 0.726 CI[0.583,0.857] n=42
RESIDUALISED probe: 0.772 CI[0.717,0.825]  AUC 0.607


In [14]:
import gc; gc.collect(); torch.cuda.empty_cache(); print("alloc GB", round(torch.cuda.memory_allocated()/1e9,2), "reserved GB", round(torch.cuda.memory_reserved()/1e9,2), "model in ns:", 'model' in globals())

alloc GB 0.01 reserved GB 0.2 model in ns: False


In [15]:
#@title 8. MOUTH: ask the Instruct sibling directly (yes/no x2 + A/B forced choice)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
itok = AutoTokenizer.from_pretrained(MODEL_INSTRUCT, token=HF_TOKEN)
ikw = dict(device_map="auto", torch_dtype=torch.float16,
           quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16), token=HF_TOKEN)
if ATTN_EAGER: ikw["attn_implementation"] = "eager"
imodel = AutoModelForCausalLM.from_pretrained(MODEL_INSTRUCT, **ikw); imodel.eval()
HAS_TMPL = bool(getattr(itok, "chat_template", None))
print("chat template:", HAS_TMPL)
def tokid_set(words):
    ids=set()
    for w in words:
        t=itok.encode(w, add_special_tokens=False)
        if t: ids.add(t[0])
    return list(ids)
YES=tokid_set(["Yes"," Yes","yes"," yes","YES"]); NO=tokid_set(["No"," No","no"," no","NO"])
A_=tokid_set(["A"," A"]); B_=tokid_set(["B"," B"])
def encode_prompt(prompt):
    if HAS_TMPL:
        enc=itok.apply_chat_template([{"role":"user","content":prompt}], add_generation_prompt=True, return_tensors="pt")
        if not torch.is_tensor(enc): enc=enc["input_ids"]
    else:
        enc=itok("[INST] "+prompt+" [/INST]", return_tensors="pt")["input_ids"]
    return enc
def first_token_probs(prompt, ids1, ids2):
    enc=encode_prompt(prompt).to(imodel.device)
    if enc.shape[1]>7500: enc=enc[:,:7500]
    with torch.no_grad():  # last-position logits only (full-vocab logits for all positions OOM on Gemma-2's 256k vocab)
        try: logits=imodel(enc, logits_to_keep=1).logits[0,-1]
        except TypeError: logits=imodel(enc, num_logits_to_keep=1).logits[0,-1]
    pr=torch.softmax(logits.float(), dim=-1)
    p1=float(sum(pr[i] for i in ids1)); p2=float(sum(pr[i] for i in ids2))
    return p1/(p1+p2+1e-9)
def clip(code, cap): return itok.decode(itok.encode(code, add_special_tokens=False)[:cap])
P1 = "Is the following Rust function vulnerable to a security bug? Answer with exactly one word: Yes or No.\n\n```rust\n{c}\n```"
P2 = "You are a Rust security expert auditing production code for memory-safety, soundness, and logic vulnerabilities. Does this function contain a security vulnerability? Answer with exactly one word: Yes or No.\n\n```rust\n{c}\n```"
ladder={}
for name,tmpl in [("zero-shot",P1),("expert",P2)]:
    ps=np.zeros(len(df))
    for i,codetxt in enumerate(df.code):
        ps[i]=first_token_probs(tmpl.format(c=clip(codetxt,MAX_TOKENS)), YES, NO)
        if i%100==0: print(name, i, len(df))
    ladder[name]=dict(auc=float(roc_auc_score(df.label,ps)), pairwise=pairwise_acc(ps), eqlen=pairwise_acc(ps,eqlen), acc=float(((ps>0.5).astype(int)==df.label).mean()))
    print(name, ladder[name])
PAB = ("Two versions of the same Rust function are shown. Exactly one contains a security vulnerability; the other is the fixed version. "
       "Which one is vulnerable? Answer with exactly one letter: A or B.\n\nVersion A:\n```rust\n{a}\n```\n\nVersion B:\n```rust\n{b}\n```")
res_ab={}
for k,cve in enumerate(pair_cves):
    g=df[df.cve==cve]; v=g.loc[g.label==1,"code"].iloc[0]; f=g.loc[g.label==0,"code"].iloc[0]
    vul_is_A=(k%2==0); a,bb=(v,f) if vul_is_A else (f,v)
    pA=first_token_probs(PAB.format(a=clip(a,PAIR_TOKENS), b=clip(bb,PAIR_TOKENS)), A_, B_)
    res_ab[cve]=1.0 if (pA>0.5)==vul_is_A else 0.0
    if k%50==0: print("A/B", k, len(pair_cves))
ab_overall=sum(res_ab.values())/len(res_ab)
mab,loab,hiab,nab = boot_ci(res_ab)
print(f"A/B forced choice: {ab_overall:.3f} CI[{loab:.3f},{hiab:.3f}]")
results["mouth"]=dict(ladder=ladder, ab=[mab,loab,hiab,nab])
del imodel; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/507 [00:00<?, ?it/s]

chat template: True
zero-shot 0 460
zero-shot 100 460
zero-shot 200 460
zero-shot 300 460
zero-shot 400 460
zero-shot {'auc': 0.5194990548204159, 'pairwise': 0.5394736842105263, 'eqlen': 0.5555555555555556, 'acc': 0.5065217391304347}
expert 0 460
expert 100 460
expert 200 460
expert 300 460
expert 400 460
expert {'auc': 0.5124480151228734, 'pairwise': 0.5, 'eqlen': 0.6111111111111112, 'acc': 0.5130434782608696}
A/B 0 228
A/B 50 228
A/B 100 228
A/B 150 228
A/B 200 228
A/B forced choice: 0.500 CI[0.434,0.566]


In [16]:
#@title 9. Row for the cross-model table + JSON
p=results["probe"]["pairwise"]; c=results["control"]["nonfix"]; t=results["temporal"]; r=results["residualised"]; ab=results["mouth"]["ab"]; L=results["mouth"]["ladder"]
print("="*80)
print(f"MODEL: {MODEL_TAG}   (base {MODEL_BASE} / instruct {MODEL_INSTRUCT}; best layer {best_layer} of {L1-1})")
print(f"{'MOUTH zero-shot / expert / A-B':40s} {L['zero-shot']['pairwise']:.3f} / {L['expert']['pairwise']:.3f} / {ab[0]:.3f}  [A/B CI {ab[1]:.3f},{ab[2]:.3f}]")
print(f"{'length floor':40s} {results['length_floor']['pairwise']:.3f}")
print(f"{'BRAIN probe (CVE pairs)':40s} {p[0]:.3f}  [{p[1]:.3f},{p[2]:.3f}]  AUC {results['probe']['auc']:.3f}")
print(f"{'BRAIN temporal (2024+)':40s} {t[0]:.3f}  [{t[1]:.3f},{t[2]:.3f}]  n={t[3]}")
print(f"{'BRAIN residualised':40s} {r[0]:.3f}  [{r[1]:.3f},{r[2]:.3f}]")
print(f"{'CONTROL non-security patches':40s} {c[0]:.3f}  [{c[1]:.3f},{c[2]:.3f}]   length rule {results['control']['len_rule']:.3f}   gap {results['control']['gap']:.3f}")
print("="*80)
json.dump(results, open(f"results_{MODEL_TAG}.json","w"), default=float)
print("===RESULTS_JSON==="); print(json.dumps(results, default=float)); print("===END_JSON===")

MODEL: Codestral-22B   (base mistralai/Codestral-22B-v0.1 / instruct mistralai/Codestral-22B-v0.1; best layer 21 of 56)
MOUTH zero-shot / expert / A-B           0.539 / 0.500 / 0.500  [A/B CI 0.434,0.566]
length floor                             0.805
BRAIN probe (CVE pairs)                  0.774  [0.721,0.827]  AUC 0.607
BRAIN temporal (2024+)                   0.830  [0.732,0.920]  n=56
BRAIN residualised                       0.772  [0.717,0.825]
CONTROL non-security patches             0.637  [0.575,0.697]   length rule 0.801   gap 0.137
===RESULTS_JSON===
{"model": "Codestral-22B", "base": "mistralai/Codestral-22B-v0.1", "instruct": "mistralai/Codestral-22B-v0.1", "layers": 56, "best_layer": 21, "probe": {"pairwise": [0.7741228070175439, 0.7214912280701754, 0.8267543859649122, 228], "auc": 0.606531190926276, "eqlen": 0.7083333333333334}, "temporal": [0.8303571428571429, 0.7321428571428571, 0.9196428571428571, 56], "control": {"nonfix": [0.6371681415929203, 0.5752212389380531, 0.6

In [17]:
import textwrap; print("\n".join(textwrap.wrap(json.dumps(results, default=lambda x: round(float(x),4)), 95)))

{"model": "Codestral-22B", "base": "mistralai/Codestral-22B-v0.1", "instruct":
"mistralai/Codestral-22B-v0.1", "layers": 56, "best_layer": 21, "probe": {"pairwise":
[0.7741228070175439, 0.7214912280701754, 0.8267543859649122, 228], "auc": 0.606531190926276,
"eqlen": 0.7083333333333334}, "temporal": [0.8303571428571429, 0.7321428571428571,
0.9196428571428571, 56], "control": {"nonfix": [0.6371681415929203, 0.5752212389380531,
0.6969026548672567, 226], "bugfix": [0.7261904761904762, 0.5833333333333334,
0.8571428571428571, 42], "len_rule": 0.8008849557522124, "gap": 0.13695466542462353},
"residualised": [0.7719298245614035, 0.7171052631578947, 0.8245614035087719, 228],
"length_floor": {"pairwise": 0.8048245614035088, "eqlen": 0.6805555555555556}, "mouth":
{"ladder": {"zero-shot": {"auc": 0.5194990548204159, "pairwise": 0.5394736842105263, "eqlen":
0.5555555555555556, "acc": 0.5065217391304347}, "expert": {"auc": 0.5124480151228734,
"pairwise": 0.5, "eqlen": 0.6111111111111112, "acc": 0.51